In [4]:
import requests                                    # HTTP library for API calls and downloading HTML pages
from bs4 import BeautifulSoup                      # HTML parser to extract clean text from RIS HTML pages
import json                                        # For parsing API responses and saving structured data
import time                                        # For adding delays between requests
import os                                          # For creating directories

os.makedirs("raw_laws", exist_ok=True)             # Create the output folder if it doesn't exist yet

# -------------------------------------------------------
# SCHRITT 1: Alle Seiten einer Suchanfrage herunterladen
# -------------------------------------------------------

def get_all_document_refs(gesetzesnummer, gesetzname):  # Function to collect all paragraph references for one law
    """
    Holt alle Paragraphen-Referenzen eines Gesetzes.
    Da RIS nur 20 pro Seite liefert, müssen wir alle Seiten durchgehen.
    """
    
    alle_refs = []                                 # Initialize empty list to collect all paragraph references
    seite = 1                                      # Start on page 1
    
    while True:                                    # Keep looping until all pages have been fetched
        print(f"  Lade {gesetzname} Seite {seite}...")  # Print progress for the current page
        
        url = "https://data.bka.gv.at/ris/api/v2.6/Bundesrecht"  # RIS API base URL
        params = {                                 # API query parameters
            "Applikation": "BrKons",               # Request consolidated federal law version
            "Gesetzesnummer": gesetzesnummer,       # Which law to fetch
            "Fassung.FassungVom": "2025-01-01",    # Law version as of Jan 1 2025
            "PageSize": 20,                        # Maximum 20 entries per page (API limit)
            "PageNumber": seite                    # Which page to fetch in this iteration
        }
        
        response = requests.get(url, params=params)  # Send the GET request
        
        if response.status_code != 200:            # If the request failed...
            print(f"  Fehler auf Seite {seite}: {response.status_code}")  # Print the error code
            break                                  # Stop the loop
        
        data = json.loads(response.text)           # Parse the JSON response body
        results = data['OgdSearchResult']['OgdDocumentResults']  # Navigate to the results section
        
        # Gesamtanzahl der Treffer auslesen
        gesamt = int(results['Hits']['#text'])     # Read the total number of paragraphs available for this law
        if seite == 1:                             # Only print the total on the first page
            print(f"  Gesamt: {gesamt} Paragraphen")  # Show total paragraph count
        
        # Einträge dieser Seite zur Gesamtliste hinzufügen
        refs = results['OgdDocumentReference']     # Get the list of paragraph references on this page
        
        # manchmal ist es eine Liste, manchmal ein einzelnes dict -> abfangen
        if isinstance(refs, list):                 # If it's already a list (multiple paragraphs)...
            alle_refs.extend(refs)                 # Add all of them to the master list
        else:                                      # If it's a single dict (only one paragraph on this page)...
            alle_refs.append(refs)                 # Wrap it in a list by appending it
        
        # Prüfen ob wir alle Seiten geladen haben
        if seite * 20 >= gesamt:                   # If we've fetched enough entries to cover the total count...
            break                                  # Stop the loop — all pages downloaded
        
        seite += 1                                 # Move to the next page
        time.sleep(0.5)                            # Wait 0.5 seconds between page requests
    
    print(f"  ✓ {len(alle_refs)} Paragraphen-Referenzen gesammelt")  # Confirm total collected
    return alle_refs                               # Return the complete list of paragraph references


# -------------------------------------------------------
# SCHRITT 2: Für jeden Paragraphen den HTML-Text laden
# -------------------------------------------------------

def extract_text_from_html(html_url):              # Function that downloads and parses one paragraph's HTML page
    """
    Lädt die HTML-Seite eines Paragraphen und extrahiert den reinen Text.
    Entfernt alle HTML-Tags und gibt nur den lesbaren Text zurück.
    """
    
    response = requests.get(html_url, timeout=10)  # Download the HTML page; give up after 10 seconds
    
    if response.status_code != 200:                # If the download failed...
        return ""                                  # Return empty string
    
    # BeautifulSoup parst das HTML
    soup = BeautifulSoup(response.text, "lxml")    # Parse the raw HTML with the lxml parser
    
    # Den Hauptinhalt finden (RIS verwendet div mit Klasse "Dokumentinhalt")
    inhalt = soup.find("div", class_="Dokumentinhalt")  # Try to find the main content div by its CSS class
    if inhalt is None:                             # If that specific div wasn't found...
        inhalt = soup.find("body")                 # Fall back to the entire page body
    if inhalt is None:                             # If even the body wasn't found (empty/broken page)...
        return ""                                  # Return empty string
    
    # Reinen Text extrahieren
    text = inhalt.get_text(separator="\n")         # Extract all visible text, using newlines between elements
    
    # Mehrfache Leerzeilen auf eine reduzieren
    zeilen = [z.strip() for z in text.splitlines()]  # Split into lines and strip whitespace from each
    text_sauber = "\n".join(z for z in zeilen if z)  # Re-join, skipping any now-empty lines
    
    return text_sauber                             # Return the cleaned plain text


# -------------------------------------------------------
# SCHRITT 3: Alles zusammenführen und speichern
# -------------------------------------------------------

def process_gesetz(gesetzname, gesetzesnummer):    # Main function: downloads all paragraphs of one law
    """
    Hauptfunktion: lädt alle Paragraphen eines Gesetzes
    und speichert sie als strukturiertes JSON.
    """
    
    print(f"\n{'='*50}")                           # Print a separator line for readability
    print(f"Verarbeite: {gesetzname}")             # Print which law is being processed
    print(f"{'='*50}")
    
    refs = get_all_document_refs(gesetzesnummer, gesetzname)  # Step 1: get all paragraph references
    
    paragraphen = []                               # Initialize list to collect finished paragraph entries
    
    for i, ref in enumerate(refs):                 # Loop through every paragraph reference
        
        # Metadaten extrahieren
        meta = ref['Data']['Metadaten']            # Navigate to the metadata section of this reference
        brkons = meta['Bundesrecht']['BrKons']     # Navigate to the federal law specific metadata
        
        paragraph_nr    = brkons.get('ArtikelParagraphAnlage', '')  # Get "§ 23" style label; default to empty if missing
        paragraphnummer = brkons.get('Paragraphnummer', '')          # Get plain number "23"; default to empty if missing
        abkuerzung      = brkons.get('Abkuerzung', gesetzname)      # Get abbreviation like "EStG 1988"; fall back to the name
        
        # HTML-URL finden
        html_url = None                            # Initialize the URL variable as empty
        
        # Dokumentliste kann dict ODER Liste sein -> abfangen
        dokumentliste = ref['Data']['Dokumentliste']  # Access the document list for this paragraph
        if isinstance(dokumentliste, list):        # If it's a list (API inconsistency)...
            dokumentliste = dokumentliste[0]       # Just take the first element
        
        # ContentReference kann dict ODER Liste sein -> abfangen
        content_ref = dokumentliste['ContentReference']  # Access the content reference section
        if isinstance(content_ref, list):          # If it's a list...
            content_ref = content_ref[0]           # Take the first element
        
        # ContentUrl kann dict ODER Liste sein -> abfangen
        urls = content_ref['Urls']['ContentUrl']   # Access the list of download URLs
        if isinstance(urls, list):                 # If multiple URLs are provided...
            for url_entry in urls:                 # Loop through each URL entry
                if url_entry.get('DataType') == 'Html':  # Look for the one marked as HTML format
                    html_url = url_entry['Url']    # Store the HTML URL
                    break                          # Stop looking once found
        elif isinstance(urls, dict):               # If only one URL is provided (as a dict, not list)...
            if urls.get('DataType') == 'Html':     # Check if it's the HTML version
                html_url = urls['Url']             # Store the URL
        
        # Falls kein HTML-Link gefunden -> überspringen
        if not html_url:                           # If no HTML URL was found at all...
            print(f"  Kein HTML-Link für {paragraph_nr}, überspringe...")  # Log the skip
            continue                               # Skip this paragraph and move to the next one
        
        # Text herunterladen
        print(f"  [{i+1}/{len(refs)}] {paragraph_nr} laden...")  # Print progress indicator
        text = extract_text_from_html(html_url)    # Step 2: download and parse the HTML page for this paragraph
        
        if text:                                   # Only save if we actually got some text back
            paragraphen.append({                   # Add a structured entry to the results list
                "gesetz": abkuerzung,              # Law name, e.g. "EStG 1988"
                "paragraph": paragraph_nr,           # Paragraph label, e.g. "§ 23"
                "paragraphnummer": paragraphnummer,  # Plain number, e.g. "23"
                "text": text,                        # The actual paragraph text content
                "quelle": f"{abkuerzung} {paragraph_nr}"  # Source citation string for RAG referencing
            })
        
        time.sleep(0.3)                            # Wait 0.3 seconds between paragraph downloads to avoid overloading RIS
    
    # Als JSON speichern
    output_path = f"raw_laws/{gesetzname}_paragraphen.json"  # Build the output file path
    with open(output_path, "w", encoding="utf-8") as f:  # Open the file for writing
        json.dump(paragraphen, f, ensure_ascii=False, indent=2)  # Save as pretty-printed JSON; keep German characters as-is
    
    print(f"✓ {len(paragraphen)} Paragraphen gespeichert in {output_path}")  # Confirm how many were saved
    return paragraphen                             # Return the list so it can be merged with other laws


# -------------------------------------------------------
# SCHRITT 4: Alle Gesetze verarbeiten
# -------------------------------------------------------

gesetze = {                                        # Dictionary of all four laws to process
    "EStG":   "10004570",                          # Einkommensteuergesetz
    "UStG":   "10004873",                          # Umsatzsteuergesetz
    "KStG":   "10004569",                          # Körperschaftsteuergesetz
    "GrEStG": "10004531"                           # Grunderwerbsteuergesetz
}

alle_paragraphen = []                              # Master list to collect paragraphs from all four laws

for name, nummer in gesetze.items():               # Loop through each law
    paragraphen = process_gesetz(name, nummer)     # Download and process all paragraphs for this law
    alle_paragraphen.extend(paragraphen)           # Add results to the master list
    time.sleep(2)                                  # Wait 2 seconds between laws as a courtesy to the RIS server

# Alles in einer großen Datei speichern -> wird später in Colab verwendet
with open("raw_laws/alle_gesetze.json", "w", encoding="utf-8") as f:  # Open the combined output file
    json.dump(alle_paragraphen, f, ensure_ascii=False, indent=2)  # Save everything in one JSON file

print(f"\n{'='*50}")
print(f"FERTIG! Gesamt: {len(alle_paragraphen)} Paragraphen")  # Print the total number of paragraphs downloaded
print(f"Gespeichert in raw_laws/alle_gesetze.json")            # Print the output file path
print(f"{'='*50}")


Verarbeite: EStG
  Lade EStG Seite 1...
  Gesamt: 188 Paragraphen
  Lade EStG Seite 2...
  Lade EStG Seite 3...
  Lade EStG Seite 4...
  Lade EStG Seite 5...
  Lade EStG Seite 6...
  Lade EStG Seite 7...
  Lade EStG Seite 8...
  Lade EStG Seite 9...
  Lade EStG Seite 10...
  ✓ 200 Paragraphen-Referenzen gesammelt
  [1/200] § 0 laden...
  [2/200] § 1 laden...
  [3/200] § 2 laden...
  [4/200] § 3 laden...
  [5/200] § 4 laden...
  [6/200] § 4a laden...
  [7/200] § 4b laden...
  [8/200] § 4c laden...
  [9/200] § 4d laden...
  [10/200] § 5 laden...
  [11/200] § 6 laden...
  [12/200] § 7 laden...
  [13/200] § 7a laden...
  [14/200] § 8 laden...
  [15/200] § 9 laden...
  [16/200] § 10 laden...
  [17/200] § 11 laden...
  [18/200] § 12 laden...
  [19/200] § 13 laden...
  [20/200] § 14 laden...
  [21/200] § 0 laden...
  [22/200] § 1 laden...
  [23/200] § 2 laden...
  [24/200] § 3 laden...
  [25/200] § 4 laden...
  [26/200] § 4a laden...
  [27/200] § 4b laden...
  [28/200] § 4c laden...
  [29/20

In [5]:
import json                                        # Standard library for JSON handling

# Alle Gesetze laden und Überblick verschaffen
with open("raw_laws/alle_gesetze.json", "r", encoding="utf-8") as f:  # Open the combined law file
    data = json.load(f)                            # Parse the JSON into a Python list

# Gesamtanzahl
print(f"Gesamt Paragraphen: {len(data)}")          # Print total number of paragraphs across all laws

# Aufschlüsselung pro Gesetz
from collections import Counter                    # Import Counter for counting occurrences
gesetze_count = Counter(p['gesetz'] for p in data) # Count how many paragraphs each law contributed
for gesetz, anzahl in gesetze_count.items():        # Loop through the counts
    print(f"  {gesetz}: {anzahl} Paragraphen")      # Print each law and its paragraph count

# Einen Beispiel-Eintrag anschauen
print(f"\n=== Beispiel-Eintrag ===")               # Section header
print(f"Gesetz: {data[10]['gesetz']}")             # Print the law name for entry #10
print(f"Paragraph: {data[10]['paragraph']}")       # Print the paragraph label (e.g., "§ 10")
print(f"Quelle: {data[10]['quelle']}")             # Print the full source citation string
print(f"Textlänge: {len(data[10]['text'])} Zeichen")  # Print the character length of the paragraph text
print(f"\nText (erste 500 Zeichen):")              # Label for the text preview
print(data[10]['text'][:500])                      # Print the first 500 characters of the paragraph text

Gesamt Paragraphen: 360
  EStG 1988: 200 Paragraphen
  UStG 1994: 60 Paragraphen
  KStG 1988: 60 Paragraphen
  GrEStG 1987: 40 Paragraphen

=== Beispiel-Eintrag ===
Gesetz: EStG 1988
Paragraph: § 6
Quelle: EStG 1988 § 6
Textlänge: 23730 Zeichen

Text (erste 500 Zeichen):
Kurztitel
Einkommensteuergesetz 1988
Kundmachungsorgan
BGBl. Nr. 400/1988 zuletzt geändert durch BGBl. I Nr. 110/2023
Bundesgesetzblatt Nr. 400 aus 1988, zuletzt geändert durch Bundesgesetzblatt Teil eins, Nr. 110 aus 2023,
Typ
BG
§/Artikel/Anlage
Paragraph/Artikel/Anlage
§ 6
Paragraph 6
Inkrafttretensdatum
22.07.2023
Abkürzung
EStG 1988
Index
32/02 Steuern vom Einkommen und Ertrag
Beachte
zum Bezugszeitraum vgl. § 124b Z 384, 423 und 427
zum Bezugszeitraum vergleiche Paragraph 124 b, Ziffer 384
